# Data Storage Layer
## Big Data Analytics Project - Mobile Device Market Analysis

This notebook covers the **Data Storage Layer** of our Big Data Analytics project.

### Objectives:
- Store data in SQL database (PostgreSQL)
- Store data in NoSQL database (MongoDB)
- Communicate with databases (Import/Export)
- Merge multiple data sources appropriately
- Exploit specific properties of each database

In [1]:
import syssys.path.append('..')import pandas as pdimport numpy as npfrom datetime import datetimeimport json# Database connectionsfrom src.db_connect import get_connectionfrom src.mongodb_connect import get_mongodb_connectionprint("SUCCESS: All imports successful!")

SyntaxError: invalid syntax (3463980307.py, line 1)

## 2. Load Raw Data

In [ ]:
# Load data from Data Source Layer
raw_df = pd.read_csv("../data/raw_merged_data.csv")

print(f"Loaded {len(raw_df)} records")
print(f"Columns: {list(raw_df.columns)}")
raw_df.head()

## 3. PostgreSQL (SQL Database)

Store structured data in PostgreSQL relational database.

In [ ]:
# Connect to PostgreSQL
pg_conn = get_connection()

if pg_conn:
    print("SUCCESS: Connected to PostgreSQL")
else:
    print("ERROR: Failed to connect to PostgreSQL")

### Create Tables and Schema

In [ ]:
if pg_conn:
    cur = pg_conn.cursor()
    
    # Create devices table
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS mobile_devices (
        id SERIAL PRIMARY KEY,
        device_name VARCHAR(255),
        brand VARCHAR(100),
        price DECIMAL(10, 2),
        screen_size DECIMAL(5, 2),
        ram INTEGER,
        storage INTEGER,
        os VARCHAR(50),
        release_date VARCHAR(20),
        category VARCHAR(50),
        source VARCHAR(50),
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """
    
    cur.execute(create_table_sql)
    pg_conn.commit()
    print("SUCCESS: Table 'mobile_devices' created")
    
    # Clear existing data (optional)
    cur.execute("TRUNCATE TABLE mobile_devices CASCADE;")
    pg_conn.commit()
    
    cur.close()

### Import Data to PostgreSQL

In [ ]:
if pg_conn:
    # Prepare data for insertion
    insert_sql = """
    INSERT INTO mobile_devices 
    (device_name, brand, price, screen_size, ram, storage, os, release_date, category, source)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    
    cur = pg_conn.cursor()
    
    # Insert records
    records_inserted = 0
    for _, row in raw_df.iterrows():
        try:
            values = (
                row.get('device_name', None),
                row.get('brand', None),
                row.get('price', None),
                row.get('screen_size', None),
                row.get('ram', None),
                row.get('storage', None),
                row.get('os', None),
                row.get('release_date', None),
                row.get('category', None),
                row.get('source', 'unknown')
            )
            cur.execute(insert_sql, values)
            records_inserted += 1
        except Exception as e:
            print(f"WARNING: Error inserting record: {e}")
            continue
    
    pg_conn.commit()
    cur.close()
    
    print(f"SUCCESS: Inserted {records_inserted} records into PostgreSQL")

### Query Data from PostgreSQL

In [ ]:
if pg_conn:
    # Export data using SQL query
    query = """
    SELECT device_name, brand, price, screen_size, ram, storage, os, category
    FROM mobile_devices
    ORDER BY price DESC
    LIMIT 10;
    """
    
    pg_df = pd.read_sql_query(query, pg_conn)
    print("Top 10 devices by price (from PostgreSQL):")
    print(pg_df)

## 4. MongoDB (NoSQL Database)

Store semi-structured and flexible data in MongoDB NoSQL database.

In [ ]:
# Connect to MongoDB
mongodb_db = get_mongodb_connection()

if mongodb_db:
    print("SUCCESS: Connected to MongoDB")
else:
    print("WARNING: MongoDB not available - using local fallback")
    print("Note: Install MongoDB or set MONGODB_URI in .env file")

### Import Data to MongoDB

In [ ]:
if mongodb_db:
    # Get collection
    devices_collection = mongodb_db['mobile_devices']
    
    # Clear existing data
    devices_collection.delete_many({})
    
    # Convert DataFrame to list of dictionaries
    documents = raw_df.to_dict('records')
    
    # Add metadata
    for doc in documents:
        doc['stored_at'] = datetime.now().isoformat()
        doc['database'] = 'mongodb'
    
    # Insert documents
    result = devices_collection.insert_many(documents)
    
    print(f"SUCCESS: Inserted {len(result.inserted_ids)} documents into MongoDB")
    print(f"Collection: {devices_collection.name}")

### Query Data from MongoDB (NoSQL Advantages)

In [ ]:
if mongodb_db:
    devices_collection = mongodb_db['mobile_devices']
    
    # Example 1: Find premium devices
    premium_devices = list(devices_collection.find({"category": "Premium"}))
    print(f"Found {len(premium_devices)} premium devices")
    
    # Example 2: Aggregate query (MongoDB aggregation pipeline)
    pipeline = [
        {"$group": {
            "_id": "$brand",
            "avg_price": {"$avg": "$price"},
            "count": {"$sum": 1}
        }},
        {"$sort": {"avg_price": -1}}
    ]
    
    brand_stats = list(devices_collection.aggregate(pipeline))
    print("\nBrand Statistics (MongoDB Aggregation):")
    for stat in brand_stats:
        print(f"  {stat['_id']}: {stat['count']} devices, Avg Price: ${stat['avg_price']:.2f}")

## 5. Save Processed Data for Next Layer

In [ ]:
# Save cleaned data for Processing Layer
if pg_conn:
    # Export from PostgreSQL
    final_query = "SELECT * FROM mobile_devices;"
    processed_df = pd.read_sql_query(final_query, pg_conn)
    
    # Save to CSV
    output_path = "../data/stored_data.csv"
    processed_df.to_csv(output_path, index=False)
    print(f"SUCCESS: Processed data saved to {output_path}")
    
    # Close connection
    pg_conn.close()
    print("PostgreSQL connection closed")

## Summary

**Data Storage Layer Complete**

- Stored data in PostgreSQL (SQL)
- Stored data in MongoDB (NoSQL)
- Imported/Exported data from both databases
- Exploited specific properties of each database
- Merged data sources appropriately

